# Data 300: Statistical and Machine Learning (Fall 2025)


# Lab: Classification Using Logistics Regression

In [1]:
## RUN THIS CELL TO GET THE RIGHT FORMATTING
import requests
from IPython.core.display import HTML
styles = requests.get("https://raw.githubusercontent.com/Harvard-IACS/2018-CS109A/master/content/styles/cs109.css").text
HTML(styles)

## The Heart Disease Dataset

The [**Heart Disease dataset**](https://www.kaggle.com/datasets/redwankarimsony/heart-disease-data) is a multivariate medical dataset used to predict the presence of heart disease. It contains demographic, clinical, and diagnostic attributes collected from multiple studies (e.g., Cleveland). The target variable is **`num`**, which indicates the presence and severity of heart disease.

- **Observations (rows):** 920 patients  
- **Variables (columns):** 16  

### Column Descriptions
- **id** – Unique identifier for each patient  
- **age** – Age of the patient (years)  
- **sex** – Sex (`Male` / `Female`)  
- **dataset** – Origin / place of study (e.g., Cleveland)  
- **cp** – Chest pain type (*typical angina, atypical angina, non-anginal, asymptomatic*)  
- **trestbps** – Resting blood pressure (mm Hg on admission)  
- **chol** – Serum cholesterol (mg/dl)  
- **fbs** – Fasting blood sugar > 120 mg/dl (`True` / `False`)  
- **restecg** – Resting electrocardiographic results (*normal, stt abnormality, lv hypertrophy*)  
- **thalch** – Maximum heart rate achieved  
- **exang** – Exercise-induced angina (`True` / `False`)  
- **oldpeak** – ST depression induced by exercise relative to rest  
- **slope** – Slope of the peak exercise ST segment (*upsloping, flat, downsloping*)  
- **ca** – Number of major vessels (0–3) colored by fluoroscopy  
- **thal** – Thalassemia status (*normal, fixed defect, reversible defect*)  
- **num** – Diagnosis of heart disease (integer; presence/absence or severity of disease)  


## Load and Explore the Heart Disease  Dataset

In the data directory, you will find the csv file for the heart disease dataset: `heart_disease_uci.csv`.

Run the two code cells below, which does the following:
- Imports the libraries that we will be using in this notebook.
- Sets `datadir = "logitData"` and uses the `os.path.join(.)` function to create a full path to the data.
- Reads in the file `heart_disease_uci.csv` and assigned it to the data frame `heart_df`.<br>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import os

In [ ]:
datadir = "logitData"
heart_path = os.path.join(datadir, "heart_disease_uci.csv")

heart_df = pd.read_csv(heart_path)

## Exploring the Data Structure

Before we begin any analysis or modeling, it’s important to **understand the structure of the dataset**. Doing this helps us:
- Identify variable types (numeric, categorical, boolean)
- Detect missing values or unusual ranges
- Recognize unexpected categories or data entry issues
- Decide what cleaning or preprocessing steps may be necessary

In the code cell below, we use: `heart_df.describe(include="all")`.

Normally, `DataFrame.describe()` only summarizes **numeric** columns (showing mean, standard deviation, quartiles, etc.). By adding `include="all"`, we tell pandas to generate summary statistics for **every column**:

- **Numeric variables** → count, mean, std, min, quartiles, max  
- **Categorical/boolean variables** → count (non-missing), number of **unique** values, the **top** (most frequent) value, and its **freq** (frequency)

👉 Run the cell below to get an overview of **all** variables in the heart disease dataset. This will help us decide what needs cleaning or transformation before moving on.


In [ ]:
heart_df.describe(include="all")

## Interpreting the Summary Statistics  

Looking at the output, there are several interesting things to notice about this dataset:

- **Sample size:** We have **920 patients** with up to 16 variables. However, some columns have missing values. The number of non-missing entries is shown in the **`count` row** of the table.  
  - To find missing values, subtract `count` from the total number of rows (920).  
  - Example: `trestbps` has 861 non-missing values → 920 – 861 = **59 missing values**.  
  - `chol` has 890 → **30 missing values**.  
  - `ca` has 309 → **611 missing values**.  
  - `thal` has 434 → **486 missing values**.  
  This means some features will require decisions about missing data.

- **Categorical variables:**  
  - `sex` has 2 categories, with **Male = 726**, Female the remainder.  
  - `dataset` has 4 study origins, with **Cleveland = 304** (the most common).  
  - `cp` (chest pain type) has 4 categories, with **asymptomatic chest pain = 496** (the largest group).  
  - `restecg`, `slope`, and `thal` are also categorical with 3 levels each.

- **Suspicious values:**  
  - `trestbps` has a **minimum of 0 mm Hg** → this is not realistic and suggests erroneous data.  
  - `chol` has a **minimum of 0 mg/dl** → also not realistic.  
  - `oldpeak` (ST depression) has a **minimum of –2.6**, which is unusual and may need to be checked.

- **Numeric ranges:**  
  - `age` ranges from 28 to 77, with a median of 54 → middle-aged to older patients.  
  - `thalach` (max heart rate) ranges from 60 to 202.  
  - `oldpeak` ranges up to 6.2, with a median around 0.5.  

- **Target variable (`num`):**  
  - Ranges from **0 to 4**.  
  - Median = 1.  
  - This indicates disease severity levels, but for many analyses we may simplify to a **binary target**: `0 = no disease`, `1 = disease present`.  

👉 These observations highlight that before modeling, we’ll need to **handle missing values (as revealed by the `count` row), check for invalid entries (e.g., zeros in blood pressure and cholesterol), and consider how to encode categorical variables**.


**Run the code cell below to get the number of missing variables more directly.**

In [ ]:
heart_df.isnull().sum()

## Handling Missing Data

When working with real-world datasets, it is common to find missing values.

In our heart disease dataset, some variables have only a few missing entries (e.g., `restecg` has 2 missing values, `chol` has 30 missing), while others have a **large number of missing values** (e.g., `ca` has 611 missing, `thal` has 486 missing).

### Should we drop all incomplete rows?
At first, it might seem easier to drop every row that contains any missing values.  
However, in this dataset that would cut the sample size nearly in half (from 920 rows down to just a few hundred). More importantly, the missingness is **not random** — for example, `ca` and `thal` come from specific medical procedures that weren’t performed on every patient. Dropping all those rows could bias our results toward a non-representative subset of patients.

### What does it mean to impute data?
**Imputation** means filling in missing values with a reasonable replacement, rather than deleting the row.  
- For numeric variables, common strategies include replacing missing values with the **mean**, **median**, or **most frequent** value.  
- For categorical variables, one option is to assign a new category (like `"missing"`) so the model can treat “not observed” as its own level.  
- Imputation is not the same as “knowing” the true value, but it allows us to use more of the dataset in a systematic way.

You can read more about how imputation works in scikit-learn here:  
👉 [SimpleImputer documentation](https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html)

### Our approach
To balance simplicity and data preservation, we will:
1. **Drop rows only if they are missing essential key data** like `age`, `sex`, or the outcome (`num`).  
   - Without these, a row cannot meaningfully contribute to analysis.  
2. **Handle the remaining missing data feature by feature:**
   - **Numeric variables** (`trestbps`, `chol`, `thalach`, `oldpeak`): impute missing values using the median.  
   - **Categorical variables** (`restecg`, `slope`, `thal`, `fbs`, `exang`): treat missing values as their own `"missing"` category so we do not lose information.  
   - **`ca`** (number of major vessels): already numeric but represents a count (0–3). We will impute missing values with a new code (e.g., `-1`) to indicate “missing.”

3. **Drop identifiers:** Let's drop the `id` column because it is just an arbitrary identifier with no relationship to heart disease.

This way, the model will know whether information about that feature was missing and can learn if missingness is predictive of the outcome.

**Run the code cell below to perform these modifications.**

In [ ]:
from sklearn.impute import SimpleImputer

# 1. Drop rows missing essential variables
heart_df = heart_df.dropna(subset=["age", "sex", "num"])

# 2a. Impute numeric variables with the median
numeric_cols = ["trestbps", "chol", "thalch", "oldpeak"]
imputer_median = SimpleImputer(strategy="median")
heart_df[numeric_cols] = imputer_median.fit_transform(heart_df[numeric_cols])

# 2b. For categorical variables, fill missing values with "missing"
categorical_cols = ["restecg", "slope", "thal", "fbs", "exang"]
for col in categorical_cols:
    heart_df[col] = heart_df[col].fillna("missing")  

# 2c. For 'ca' (number of vessels), use -1 as a code for missing
heart_df["ca"] = heart_df["ca"].fillna(-1)

# 3. Drop 'id' column
heart_df = heart_df.drop(columns=["id"])

# Quick check of remaining missing values
print(heart_df.isnull().sum())

Note that we no longer have (explict) missing values! 
**Run the code cell below to see the data types that were assigned by Pandas.**

In [ ]:
print(heart_df.dtypes)

As we can see from the output, Pandas correctly assigned the quantitative variables (`age`, `trestbps`, `chol`, `thalch`, `oldpeak`, `ca`) to numeric types (`int64` or `float64`). However, it set several categorical variables (e.g., `sex`, `dataset`, `cp`, `restecg`, `slope`, `thal`) to type `object`.  

- Recall that in Pandas, an `object` type is a general placeholder for text data, often used for categorical variables stored as strings.  
- A `category` type, on the other hand, is specifically designed for categorical data. It stores values more efficiently and makes it explicit that the variable is categorical rather than numeric.  
- The `ca` column (number of major vessels) should be an integer type, but because it has missing values we will use `Int64` (pandas’ nullable integer type).  

Finally, we need to pay special attention to the outcome variable. The column `num` in this dataset takes values from **0 to 4**, which indicate different levels of heart disease severity:  
- `0` = no disease  
- `1–4` = increasing severity of heart disease  

For our purposes, we are interested in **predicting whether or not a patient has heart disease at all**. This means we should collapse the severity levels into a single category:  
- `0` → `no disease`  
- `1–4` → `disease present`  

We will create a new binary target variable called **`heart_disease`**, where `0 = no disease` and `1 = disease present`. This binary outcome is required for logistic regression, which models the probability of a “yes/no” event.

To keep everything consistent—and to ensure categorical and numeric variables are treated appropriately—we will make the following changes:  

- Convert `sex`, `dataset`, `cp`, `restecg`, `slope`, `thal` → `category`  
- Convert `ca` → `Int64`  
- Create new `heart_disease` variable from `num` (binary outcome for logistic regression)
- Drop the `num` column to prevent data leakage

Note that **data leakage** happens when information from outside the training data sneaks into the model in a way that gives it an unfair advantage and inflates performance. Since num is just another version of our target, keeping it would let the model "cheat," so we drop it before training.

**Please run the code cell below to perform these conversions.**

In [ ]:
# Convert categorical features stored as object → category
categorical_cols = ["sex", "dataset", "cp", "restecg", "slope", "thal", "fbs", "exang"]
heart_df[categorical_cols] = heart_df[categorical_cols].astype("category")

# Convert 'ca' to integer type
heart_df["ca"] = heart_df["ca"].astype("Int64")

# Create new binary target variable from 'num'
# 0 = no disease, 1 = disease present (num > 0)
heart_df["heart_disease"] = (heart_df["num"] > 0).astype(int)

# Drop the 'num' column to prevent data leakage
heart_df = heart_df.drop(columns=["num"])

# Check results
heart_df.dtypes

All of the categories now look correct! **Run the code cell below to see the distributions of the categorical variables.**

In [ ]:
# Show a summary of all categorical and boolean variables
print("=== Categorical & Boolean Variables Summary ===")

# Identify categorical and boolean columns
categorical_columns = heart_df.select_dtypes(include=["category"]).columns

for column in categorical_columns:
    # Count how many unique values this column has
    num_unique = heart_df[column].nunique(dropna=False)  # include NaN if present
    
    print(f"\n{column} (Unique values = {num_unique})")
    # Show frequency counts of each category
    print(heart_df[column].value_counts(dropna=False))


### Exploratory Data Analysis (EDA)

**Exploratory Data Analysis (EDA)** is the process of looking at a dataset before building any models.  
It helps us:
- Understand what variables look like (their distributions, categories, ranges).  
- Spot potential issues such as missing values, outliers, or class imbalance.  
- Start forming ideas about which variables may be related to our target.  

EDA should *always* be the first step before modeling. Here, we’ll just do a couple of basic checks, but in a real-world project you would want to dig much deeper.  

➡️ Run the code cell below to see boxplots of **Age** by Heart Disease Status and **Cholesterol** by Heart Disease Status.


In [ ]:
sns.boxplot(x="heart_disease", y="age", data=heart_df)
plt.title("Age by Heart Disease Status")
plt.show()

sns.boxplot(x="heart_disease", y="chol", data=heart_df)
plt.title("Cholesterol by Heart Disease Status")
plt.show()

### Interpreting the Boxplots

- **Age by Heart Disease Status:**  
  The boxplots show that individuals with heart disease (1) tend to be slightly older on average than those without (0).  
  The median age is higher for the heart disease group, and the overall spread is a bit wider. There are a number of outliers corresponding to younger patients with heart disease. Overall, these boxplots suggest that age is an important factor related to heart disease.

- **Cholesterol by Heart Disease Status:**  
  The cholesterol levels have a large spread in both groups, with many outliers, both young and older patients, in the group without heart disease.
  Interestingly, the median cholesterol levels are not dramatically different between the two groups, meaning cholesterol alone may not be as strong a predictor as age.  
  However, it could still be useful when combined with other variables.

**Takeaway:**  
EDA helps us see which variables might have meaningful differences between groups (like age) and which ones might need to be used in combination with other features (like cholesterol) for the model to find patterns.  


### Chest Pain Type and Heart Disease

Next, let’s look at the relationship between **chest pain type (cp)** and **heart disease status**.  
This will give us a sense of how a categorical variable is distributed across the two groups (no heart disease vs. heart disease).  

➡️ Run the code cell below to see a histogram (countplot) of **Chest Pain Type** with colors showing **Heart Disease Status**.


In [ ]:
sns.countplot(x="cp", hue="heart_disease", data=heart_df)
plt.title("Chest Pain Type by Heart Disease Status")
plt.show()

### Interpreting the Chest Pain Type Plot

- The plot shows how different chest pain types (`cp`) are distributed for patients **with** and **without** heart disease.  
- We can see that:
  - **Asymptomatic chest pain** is much more common in patients with heart disease.  
  - **Atypical and non-anginall angina** are more frequent in patients without heart disease.  
  - **Typical anginal pain** is seen in both groups, but less dominant.  

**Takeaway:**  
Chest pain type is strongly associated with heart disease status.  
This suggests it will likely be an important predictor in our logistic regression model.  
```   ​:contentReference[oaicite:0]{index=0}​


### Slope of ST Segment and Heart Disease

Now let’s examine the **slope of the ST segment (slope)** and how it relates to heart disease status.  
This is another categorical variable that can provide insight into whether certain slope patterns are more common among patients with heart disease.  

➡️ Run the code cell below to see a histogram (countplot) of **Slope of ST Segment** with colors showing **Heart Disease Status**.


In [ ]:
sns.countplot(x="slope", hue="heart_disease", data=heart_df)
plt.title("Slope of ST Segment by Heart Disease Status")
plt.show()

### Interpreting the Slope of ST Segment Plot

- The plot shows the distribution of **slope types** for patients with and without heart disease.  
- We can see that:  
  - The **flat slope** is much more common among patients with heart disease.  
  - The **upsloping slope** is more common in patients without heart disease.  
  - The **downsloping slope** occurs less often overall, but appears more frequently in patients with heart disease.
  - There were quite a few missing values in both groups of patients. 

**Takeaway:**  
The slope of the ST segment appears to be related to heart disease risk, with a **flat slope** being a particularly strong indicator of heart disease presence.  
```   ​:contentReference[oaicite:0]{index=0}​


### Correlation Heatmap of Numeric Variables

Now we are going to create a **correlation heatmap** of the numeric variables in our dataset.  

Why does this matter?  
- Correlation tells us how strongly two variables are related to each other.  
- Looking at correlations helps us see which features might be most related to our target (`heart_disease`).  
- It also helps us spot potential problems, like features that are highly correlated with each other (multicollinearity), which can cause issues for models such as logistic regression.  

➡️ Run the code cell below to generate the heatmap.


In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(heart_df.corr(numeric_only=True), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Variables)")
plt.show()

### Interpreting the Correlation Heatmap

- The heatmap shows the pairwise correlations among all the **numeric variables** in the dataset.  
- Key observations:  
  - **`heart_disease`** is positively correlated with `oldpeak` and negatively correlated with `thalch` (maximum heart rate achieved).  
    - This means higher ST depression and lower maximum heart rate are linked to greater heart disease risk.  
  - **`age`** has a mild positive correlation with `heart_disease`, reinforcing what we saw in the boxplots.  
  - Some variables (like `trestbps` and `chol`) show weak correlations with `heart_disease`, suggesting they may not be strong predictors on their own.  
- The diagonal is always `1.0` because each variable is perfectly correlated with itself.

**Takeaway:**  
Correlation analysis gives us a first look at how numeric variables relate to heart disease and to each other.  
This helps us build intuition about which features may be more important for prediction, and reminds us to be mindful of overlapping information between features.
```   ​:contentReference[oaicite:0]{index=0}​


### Wrapping Up EDA

That concludes the exploratory data analysis (EDA) we will do here.  
In a real project, you would likely explore many more aspects of the data — for example, looking at outliers, testing different feature transformations, or exploring interactions between variables.  

For our purposes, this basic EDA gave us enough insight into the dataset to move forward.  
We are now ready to begin **building our logistic regression model**.


## Creating a Hold-Out Test Set

Before we start building logistic regression models, we want to set aside a **hold-out test set**.  
The idea is simple:  
- We train the model on part of the data (the **training set**)  
- We evaluate the model’s predictive performance on the **test set**, which the model has never seen before  

### Why is this a good idea?
- If we evaluate a model on the same data we used to fit it, the results are overly optimistic — the model may just be “memorizing” patterns rather than learning general relationships.  
- By keeping a separate test set, we can assess how well the model is likely to perform on **new, unseen data**.  
- This helps us catch problems like **overfitting**, where the model performs well on training data but poorly on new data.  

### How do we do this?
We will use the `train_test_split` function from **scikit-learn** to randomly divide our data into training and test sets. For example:

<code>
# Split into 80% training data and 20% testing data
train_df, test_df = train_test_split(heart_df, test_size=0.2, random_state=42)
</code>

- `train_df` → used to fit our logistic regression models in statsmodels

- `test_df` → used only at the end to evaluate predictive performance (confusion matrix, ROC-AUC, etc.)

- `random_state=42` → sets a random seed so the split is reproducible; each time you run the code, you’ll get the same train/test division.

👉 This way, we can compare different models on the training set and then use the test set to see which model generalizes best.

**Run the code cell below to create the training and testing sets.**


In [ ]:
from sklearn.model_selection import train_test_split

# Split into 80% training data and 20% testing data
train_df, test_df = train_test_split(heart_df, test_size=0.2, random_state=42)

### Checking Train/Test Split

Run the code cell above to verify the sizes of the training and testing sets.  
This is a good habit to confirm that the split worked as expected (about 80% of the data in the training set and 20% in the testing set).


In [ ]:
# Check the size of the train and test sets
print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

### Interpreting the Train/Test Split Sizes

Our dataset originally had **920 rows**.  
- With an 80/20 split, we expect about 80% of the data (736 rows) in the **training set** and about 20% (184 rows) in the **testing set**.  
- Both sets have **15 columns**, since we are keeping all the same features and the target variable in each.  

These numbers make sense because 736 + 184 = 920, which confirms that the split worked exactly as intended.


### Logistic Regression with `statsmodels`

We are going to use the **`logit` function** from the `statsmodels` package to build our logistic regression models.  
This function allows us to specify formulas in a way that looks very similar to R, which makes the code easy to read.  

➡️ Run the code cell below to import the `logit` function.

In [ ]:
from statsmodels.formula.api import logit

### Modeling Plan

Our plan is to build a series of logistic regression models:  
1. **Start simple** with a basic model using only the numeric predictors.  
2. **Add more sophistication** by including important categorical variables.  
3. **Compare models** to see how performance and interpretability change.  

We’ll begin with the **numeric-only model**, which will serve as our baseline.  

➡️ Run the code cell below to create the model and print the output.


In [ ]:
# Model 1: Logistic regression with only numeric predictors
formula_numeric_only = "heart_disease ~ age + trestbps + chol + thalch + oldpeak + ca"
model_numeric_only = logit(formula=formula_numeric_only, data=train_df).fit()

# Print summary
print(model_numeric_only.summary())

### Interpreting the Numeric-Only Model

- **Strong predictors:**  
  - `thalch` (maximum heart rate achieved) – negative coefficient, significant. Lower max heart rate is linked to greater heart disease risk.  
  - `oldpeak` (ST depression) – positive coefficient, significant. Higher ST depression is linked to greater risk.  
  - `ca` (number of vessels) – positive coefficient, significant. More blocked vessels corresponds to higher risk.  
  - `chol` – negative coefficient, significant. Surprisingly, higher cholesterol is associated with lower risk in this dataset, which may reflect quirks of the sample or relationships with other variables.  
- **Not significant here:** `age` and `trestbps` (resting blood pressure) did not show strong unique effects once other variables were included.

---

### What is Pseudo R²?

In linear regression we use **R²** to measure how much variation in the outcome is explained by the model.  
For logistic regression, we don’t have a true R², but we use **Pseudo R²** as a rough analogue.  

- **Interpretation:** A Pseudo R² of about 0.25 means that the numeric variables explain roughly 25% of the variation in heart disease risk.  
- It’s not directly comparable to R² from linear regression, but it is useful for **comparing models**: a higher Pseudo R² generally means a better fit.

---

### Why Keep Age and Trestbps?

Even though `age` and `trestbps` were not statistically significant in this model:  
- They are **well-established medical risk factors** for heart disease, so dropping them without discussion could mislead us.  
- They may become more useful in combination with categorical variables (e.g., chest pain type or sex) in later models.  
---

**Takeaway:** Even with just the numeric variables, our model can capture meaningful signals about heart disease risk, though there’s plenty of room to improve by adding categorical predictors.


### Assessing Model Fit with Performance Metrics

Now that we’ve fit our **baseline logistic regression model**, we need to evaluate how well it performs on **new data**.  
That’s why earlier we set aside a **testing set** — data that the model has not seen during training.  
By using this holdout set, we get a more honest estimate of how well the model might perform on future, unseen data.

We’ll use the performance metrics we’ve already learned about:  
- **Confusion matrix** (TN, FP, FN, TP)  
- **Accuracy, Sensitivity, Specificity**  
- **ROC curve and AUC**  

In Python, these come from the **`sklearn.metrics`** package:  
- `confusion_matrix` and `accuracy_score` help us summarize classification outcomes.  
- `roc_auc_score` and `roc_curve` let us compute and visualize the ROC curve and AUC.  
- We’ll also use **matplotlib** to plot the ROC curve.
- Uses the model to get **predicted probabilities** (`model.predict(test_df)`) and converts them to **predicted labels** with a 0.5 threshold.

➡️ Run the code cell below to compute these metrics for our baseline model, evaluated on the **test set**.


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score, roc_curve

# True values
y_true = test_df["heart_disease"]

# Predicted probabilities and labels
y_pred_prob = model_numeric_only.predict(test_df)
y_pred = (y_pred_prob >= 0.5).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
print("Confusion Matrix:")
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print("Accuracy:", accuracy)

# Sensitivity (Recall for positives)
sensitivity = tp / (tp + fn)
print("Sensitivity:", sensitivity)

# Specificity (Recall for negatives)
specificity = tn / (tn + fp)
print("Specificity:", specificity)

# AUC
auc = roc_auc_score(y_true, y_pred_prob)
print("AUC:", auc)

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate (1 - Specificity)")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve (Model 1: Numeric Only)")
plt.legend()
plt.show()


### Interpreting the Baseline Model Performance

Our confusion matrix and performance metrics on the **test set** are:

- **Confusion Matrix:** TN=55, FP=20, FN=32, TP=77  
- **Accuracy:** ~0.72  
- **Sensitivity (Recall for positives):** ~0.71  
- **Specificity (Recall for negatives):** ~0.73  
- **AUC:** ~0.81  

**How to interpret this:**  
- The model correctly classifies about **72%** of patients overall (accuracy).  
- Sensitivity (~71%) shows that it detects most patients with heart disease, but it still misses some (32 false negatives).  
- Specificity (~73%) means it’s also reasonably good at identifying those without heart disease, though it makes some false positive errors (20).  
- The **AUC of 0.81** indicates the model has good discriminative ability — it can separate patients with and without heart disease better than chance.

**ROC Curve:**  
The ROC curve plots **sensitivity (true positive rate)** against **1 – specificity (false positive rate)** at all possible thresholds.  
Our curve is well above the diagonal “no-skill” line, showing the model does meaningfully better than random guessing across different thresholds.

**Takeaway:**  
This baseline numeric-only model provides a reasonable first step, but there is room for improvement. Adding categorical predictors (like sex and chest pain type) may boost performance further.


### Model 2: Adding Key Categorical Predictors

Next, we’ll extend the numeric-only baseline by adding two medically meaningful categorical variables: **`sex`** and **`cp` (chest pain type)**.  
This makes sense because our EDA suggested strong associations between chest pain patterns and heart disease, and sex is a well-known risk factor.  
By starting with just a couple of categorical predictors, we keep the model interpretable while showing how categorical information can boost predictive power.

➡️ Run the code cell below to fit **Model 2 (numeric + `sex` + `cp`)** and print the results.


In [ ]:
# Model 2: Logistic regression with numeric + sex + chest pain type
formula_numeric_sex_cp = "heart_disease ~ age + trestbps + chol + thalch + oldpeak + ca + sex + cp"
model_numeric_sex_cp = logit(formula=formula_numeric_sex_cp, data=train_df).fit()

print(model_numeric_sex_cp.summary())


### Interpreting Model 2: Numeric + Sex + Chest Pain Type

- **Overall model fit:**  
  - Pseudo R² increased to **0.38** (compared to 0.25 for the numeric-only model).  
  - This means the model now explains substantially more variation in heart disease status.  
  - The likelihood ratio test is highly significant (p < 0.001), confirming the model provides strong explanatory power.

- **Key new predictors:**  
  - **Sex:** Being male is strongly associated with higher odds of heart disease (positive coefficient, p < 0.001).  
  - **Chest pain type (`cp`):**  
    - All three chest pain categories (atypical angina, non-anginal pain, typical angina) show **negative coefficients** compared to the baseline (`asymptomatic`).  
    - This means that patients with angina-related chest pain types are **much less likely** to have heart disease than those with asymptomatic pain, which matches what we saw in the earlier countplots.

- **Other predictors:**  
  - `oldpeak`, `ca`, `thalch`, and `chol` remain strong predictors, as they were in the numeric-only model.  
  - `age` and `trestbps` are still not significant, but we keep them for consistency and potential interactions with categorical features.

---

**Takeaway:** Adding just two categorical variables — sex and chest pain type — made the model much stronger.  

### Model 3: Add the Remaining Categorical Predictors (Full Model)

For our third model, we’ll extend Model 2 by adding the **remaining categorical variables**:
`dataset`, `restecg`, `slope`, `thal`, `fbs`, and `exang`.  
This creates a “full” model that combines all numeric predictors with every categorical feature we’ve prepared.  
It’s useful for seeing the *upper bound* of explainability with this feature set and for comparing how much additional signal the extra categories provide (versus the added complexity).

➡️ Run the code cell below to fit the full model and print the summary.


In [ ]:
# Model 3: Numeric predictors + sex + cp + all remaining categorical features
formula_numeric_allcats = (
    "heart_disease ~ age + trestbps + chol + thalch + oldpeak + ca "
    "+ sex + cp + dataset + restecg + slope + thal + fbs + exang"
)

model_numeric_allcats = logit(formula=formula_numeric_allcats, data=train_df).fit()

print(model_numeric_allcats.summary())

### Interpreting Model 3: Full Model (Numeric + All Categorical)

When fitting the full model, `statsmodels` gave us a **convergence warning**.  
This isn’t unusual when adding lots of categorical predictors, especially if some categories are rare.  
We’ll still look at the results to see what they show.

- **Overall fit:** The Pseudo R² increased again — it’s now **higher than 0.38 from Model 2** (and much higher than 0.25 from the numeric-only Model 1).  
  This means that, even with the complexity and noise, adding the full set of categorical variables explains more of the variation in heart disease risk.  
- **Key predictors:** Many of the strong predictors from earlier models (`sex`, `cp`, `oldpeak`, `ca`, `thalch`) remain highly significant.  
- **New categorical effects:** Some additional categorical variables contribute useful information. For example, certain levels of `slope` and `thal` show significant differences in heart disease risk, which aligns with what we observed in EDA (flat slopes being more risky, for instance).  
- **Non-significant predictors:** Several added variables (like some `dataset` categories or rare levels in `restecg`, `thal`, and `fbs`) don’t appear to add much explanatory power and may just add noise.

---

### What’s Next?

At this stage, the model is very complex and the output is long, making it harder to interpret.  
One option is to **simplify** by removing non-significant variables, which may give us a cleaner, more stable model.  

➡️ For **Model 4**, we will keep the predictors that consistently matter across models —  
`sex`, `cp`, `oldpeak`, `ca`, `thalch`, `chol`, and `slope` —  
and **drop weaker or unstable predictors** such as `dataset`, `restecg`, and `fbs`.  

This will let us compare whether a simpler model performs just as well (or even better) while being easier to interpret.

**Run the code cell below to fit this new model.**


In [ ]:
# Model 4: Core predictors (numeric + key categorical)
formula_numeric_core = "heart_disease ~ chol + thalch + oldpeak + ca + sex + cp + slope"
model_numeric_core = logit(formula=formula_numeric_core, data=train_df).fit()

print(model_numeric_core.summary())

### Interpreting Model 4: Core Predictors

In Model 4, we kept the most important predictors identified in earlier models:  
**`chol`, `thalch`, `oldpeak`, `ca`, `sex`, `cp`, and `slope`.**

- **Overall fit:**  
  - Pseudo R² = **0.39**, which is **slightly higher than Model 3 (≈0.38)** and much better than Model 2 (0.38) and Model 1 (0.25).  
  - This shows that by removing weaker variables, we improved model stability and interpretability *without losing predictive power*.

- **Key predictors (still significant):**  
  - **Sex (Male):** Strongly increases odds of heart disease.  
  - **Chest pain type (`cp`):** Non-asymptomatic chest pain types are strongly protective compared to asymptomatic.  
  - **Cholesterol, max heart rate (`thalch`), ST depression (`oldpeak`), and vessels (`ca`)** all remain significant with expected directions.  
  - **Slope:** The flat slope is marginally significant (p ≈ 0.07), consistent with what we saw in EDA.

- **What changed:**  
  - Removing less informative variables (`dataset`, `restecg`, `fbs`, and rare categories) gave us a more parsimonious model.  
  - Coefficients on the key predictors are stable, which builds confidence in their importance.

---

**Takeaway:**  
Model 4 strikes a good balance — it’s simpler and easier to interpret, yet it actually performs slightly better (higher Pseudo R²) than the full model.  
This demonstrates the value of **feature selection**: including too many predictors can add noise without improving results.
```   ​:contentReference[oaicite:0]{index=0}​


### Comparing Models on the Test Set

Now that we’ve trained our models, let’s compare their performance **on the holdout test set** using the same metrics: Accuracy, Sensitivity, Specificity, and AUC.

> **Note:** We’re **skipping Model 3 (Full Model)** in this table because it didn’t converge cleanly.  
> Comparing a model that failed to converge can be misleading: its coefficient estimates may be unstable and its predicted probabilities unreliable. It’s better practice to compare only **stable, well-fitted models**.

#### What the code below does

1. **Helper function `evaluate_model(...)`:**  
   - Takes a fitted model, a human-readable name, and the `test_df`.  
   - Uses the model to get **predicted probabilities** (`model.predict(test_df)`) and converts them to **predicted labels** with a 0.5 threshold.  
   - Computes the **confusion matrix** and extracts TN, FP, FN, TP.  
   - Calculates **Accuracy**, **Sensitivity** (recall for the positive class), **Specificity** (recall for the negative class), and **AUC** (area under the ROC curve).  
   - Returns a small dictionary with the model name and its metrics.

2. **Evaluate each model:**  
   - We call `evaluate_model` for **Model 1 (Numeric Only)**, **Model 2 (Numeric + Sex + CP)**, and **Model 4 (Core Predictors)**.  
   - We leave Model 3 commented out due to the convergence issue.

3. **Create a comparison table:**  
   - We collect the results into a list, convert it into a **Pandas DataFrame**, and print it for a clean, side-by-side summary.

➡️ **Run the code cell below to generate the comparison table for Models 1, 2, and 4.**


In [ ]:
def evaluate_model(model, model_name, test_df):
    # Predict probabilities and labels
    y_true = test_df["heart_disease"]
    y_pred_prob = model.predict(test_df)
    y_pred = (y_pred_prob >= 0.5).astype(int)
    
    # Confusion matrix components
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    # Metrics
    accuracy = accuracy_score(y_true, y_pred)
    sensitivity = tp / (tp + fn)   # recall for positives
    specificity = tn / (tn + fp)
    auc = roc_auc_score(y_true, y_pred_prob)
    
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "AUC": auc
    }

# Evaluate all models
results = []
results.append(evaluate_model(model_numeric_only, "Model 1: Numeric Only", test_df))
results.append(evaluate_model(model_numeric_sex_cp, "Model 2: Numeric + Sex + CP", test_df))
# results.append(evaluate_model(model_numeric_allcats, "Model 3: Full Model", test_df))  # may fail convergence
results.append(evaluate_model(model_numeric_core, "Model 4: Core Predictors", test_df))

# Convert to DataFrame for display
import pandas as pd
results_df = pd.DataFrame(results)
print(results_df)

### Comparing Model Performance

Here is the summary of test set performance across our models:

| Model                       | Accuracy | Sensitivity | Specificity | AUC   |
|------------------------------|----------|-------------|-------------|-------|
| Model 1: Numeric Only        | ~0.72    | ~0.71       | ~0.73       | 0.81  |
| Model 2: Numeric + Sex + CP  | ~0.78    | ~0.78       | ~0.79       | 0.87  |
| Model 4: Core Predictors     | ~0.79    | ~0.80       | ~0.79       | 0.87  |

**What we see:**
- Moving from **Model 1 → Model 2** leads to a clear improvement in every metric. Adding sex and chest pain type gives the model a significant boost.  
- **Model 4** performs slightly better than Model 2 in terms of accuracy and sensitivity, though AUC is almost identical.  
- All models have reasonably balanced sensitivity and specificity (no huge trade-off between detecting positives and negatives).  

**Which model is best?**  
- **Model 4** edges out the others in accuracy and sensitivity while remaining relatively simple (fewer predictors than the full model, which didn’t converge).  
- That said, **Model 2** is also a strong choice: it’s simpler and still shows a large improvement over the numeric-only baseline.  

**What else could be done (beyond today):**
- Try **regularization** (e.g., Lasso logistic regression) to handle correlated predictors.  
- Explore **cross-validation** instead of a single train/test split for more robust estimates.  
- Consider **feature engineering** (combining variables or adding interactions).  
- Experiment with **other classification algorithms** (decision trees, random forests, gradient boosting).  

For now, we’ve seen how thoughtful model building — starting simple, adding predictors, and then simplifying again — can improve performance while keeping models interpretable.


### Using the Model for Prediction

Now that we’ve compared our models and chosen the best one (**Model 4: Core Predictors**), let’s see how it can be used to make an actual classification.  

We’ll create a small **DataFrame with a single patient’s information**, using the same variables included in Model 4.  
The model will then give us:  
- A **predicted probability** of heart disease (between 0 and 1).  
- A **predicted class** (0 = no disease, 1 = disease), based on a cutoff of 0.5.  

➡️ Run the code cell below to see how the model classifies an example patient.


In [ ]:
# Example patient data (must include the same variables used in Model 4)
example_patient = pd.DataFrame({
    "chol": [240],             # cholesterol
    "thalch": [150],           # max heart rate achieved
    "oldpeak": [1.2],          # ST depression
    "ca": [1],                 # number of major vessels
    "sex": ["Male"],           # categorical
    "cp": ["asymptomatic"],    # categorical
    "slope": ["flat"]          # categorical
})

# Use the fitted Model 4 to predict probability
pred_prob = model_numeric_core.predict(example_patient)[0]
pred_class = int(pred_prob >= 0.5)

print("Predicted probability of heart disease:", round(pred_prob, 3))
print("Predicted class (1 = disease, 0 = no disease):", pred_class)